# Embodied Agents and Robotics

**Level:** Advanced · **Time:** 60 min

When an agent controls a robot, a hallucination can cause physical damage. 

In this notebook, we will simulate two critical robotics patterns:
1. **The Safety Supervisor:** A deterministic firewall that overrides the LLM when physical force limits are exceeded.
2. **Sensor Feedback Loops:** An agent that relies on physical weight sensors, rather than its own internal assumptions, to verify task completion.

---
## Pattern 1: The Safety Supervisor

An LLM commands a robotic arm to move downwards to grasp a box. However, the LLM miscalculated the depth. As the arm moves, it collides with the table, causing the physical torque sensors to spike. The hard-coded Safety Supervisor instantly intercepts the movement, overriding the LLM and saving the hardware.

In [1]:
import time

# Simulated Hardware State
MAX_SAFE_TORQUE_NEWTONS = 15.0

def low_level_controller(target_z_coordinate: float):
    print(f"\n[Controller] Initiating movement to Z={target_z_coordinate}m...")
    
    # Simulate movement loop (1000Hz in reality, simulated here with steps)
    current_z = 1.0
    current_torque = 2.0
    
    while current_z > target_z_coordinate:
        current_z -= 0.1
        current_torque += 3.5 # Torque increases as we push down
        
        print(f"  > [Hardware] Arm at Z={current_z:.1f}m | Joint Torque: {current_torque:.1f} N")
        
        # 🚨 THE SAFETY SUPERVISOR 🚨
        # This check runs deterministically, completely ignoring the LLM's goal.
        if current_torque > MAX_SAFE_TORQUE_NEWTONS:
            print(f"\n🚨 [SAFETY SUPERVISOR] CRITICAL OVERRIDE 🚨")
            print(f"Torque exceeded limit ({current_torque:.1f}N > {MAX_SAFE_TORQUE_NEWTONS}N). Collision detected!")
            print("ABORTING MOVEMENT. CUTTING MOTOR POWER.")
            return False
            
        time.sleep(0.5)
        
    print("\n[Controller] Movement complete.")
    return True

def mock_vla_model(instruction: str):
    print(f"[VLA Model] Instruction received: '{instruction}'")
    print("[VLA Model] Calculating semantic goal...")
    # Hallucination! The table is at Z=0.5m, but the LLM targets Z=0.1m
    return {"action": "move_arm", "target_z": 0.1}

# 1. The LLM decides what to do
semantic_goal = mock_vla_model("Grasp the box on the table.")

# 2. The physical layer attempts it, heavily supervised
if semantic_goal["action"] == "move_arm":
    low_level_controller(semantic_goal["target_z"])


[VLA Model] Instruction received: 'Grasp the box on the table.'
[VLA Model] Calculating semantic goal...

[Controller] Initiating movement to Z=0.1m...
  > [Hardware] Arm at Z=0.9m | Joint Torque: 5.5 N


  > [Hardware] Arm at Z=0.8m | Joint Torque: 9.0 N


  > [Hardware] Arm at Z=0.7m | Joint Torque: 12.5 N


  > [Hardware] Arm at Z=0.6m | Joint Torque: 16.0 N

🚨 [SAFETY SUPERVISOR] CRITICAL OVERRIDE 🚨
Torque exceeded limit (16.0N > 15.0N). Collision detected!
ABORTING MOVEMENT. CUTTING MOTOR POWER.


---
## Pattern 2: Closed-Loop Sensor Feedback

An agent is instructed to place a 5kg package into a bin. It executes the motion path. However, in an Open-Loop system, it would just assume success. In a robust Closed-Loop system, it checks the physical scale in the bin to verify the weight increased. If the package slipped, it replans.

In [2]:
def check_bin_weight_sensor():
    # Simulate a sensor reading. The package slipped and fell on the floor!
    return 0.0 # kg

def execute_pick_and_place():
    print("\n[VLA Model] Executing grasp and place trajectory...")
    print("[Controller] Moving to bin...")
    print("[Controller] Releasing gripper...")
    
    # The action finished, but did it succeed?
    print("\n[System] Verifying task completion via physical sensors...")
    bin_weight = check_bin_weight_sensor()
    
    if bin_weight < 4.0: # Expecting ~5kg
        print(f"❌ [Feedback] Bin scale reads {bin_weight}kg. The package is missing!")
        print("[VLA Model] Task failed. Re-evaluating camera feed to locate dropped package and replan...")
        return False
        
    print("✅ [Feedback] Bin scale confirmed weight increase. Task successful.")
    return True

execute_pick_and_place()



[VLA Model] Executing grasp and place trajectory...
[Controller] Moving to bin...
[Controller] Releasing gripper...

[System] Verifying task completion via physical sensors...
❌ [Feedback] Bin scale reads 0.0kg. The package is missing!
[VLA Model] Task failed. Re-evaluating camera feed to locate dropped package and replan...


False